# Notebook 06 — Lower learning rate, with epoch accuracy checks

Test a fresh LoRA adapter at **5e-5**, versus the first adapter's **2e-4**. This is a hypothesis, not a promised maths improvement. Keep training data, seed, three epochs, effective batch 32, LoRA rank/targets, optimizer, and loss-based checkpoint selection fixed.

**Kaggle:** enable Internet and a T4 GPU; attach the saved outputs of **Notebook 01 and Notebook 05** (folders or ZIPs, not just `.ipynb` files). The first adapter is compared through Notebook 05's verified saved predictions; its weights are not required. One GPU is used.

Generate answers to the same **50 validation questions, three-shot**, before training, after each epoch, and for the loss-selected adapter: **250 full-run predictions**, plus six smoke predictions. All generated metrics are observational: they never choose a checkpoint or trigger additional epochs. No test questions are generated. This small, previously inspected development sample cannot establish final generalization.

Budget roughly **60–90 minutes** on a T4, based on the earlier 48-minute training run plus generation and setup; this is an estimate. Retain the original run and all new outputs.


## 1. Fetch the shared code

The launcher records the exact GitHub commit. Training logic and settings live in the repository. Keep the original data bundle fixed throughout the experiment.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

REPOSITORY = "https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git"
PROJECT_REF = os.environ.get("MATH_PROJECT_REF", "main")
if PROJECT_REF.startswith("-"):
    raise ValueError("Use a branch, tag, or full commit SHA as MATH_PROJECT_REF")
PROJECT = Path(tempfile.mkdtemp(prefix="math-project-"))
subprocess.run(["git", "init", "--quiet", str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "remote", "add", "origin", REPOSITORY], check=True)
subprocess.run(["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", PROJECT_REF], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "--quiet", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPOSITORY)
print("Code commit:", CODE_COMMIT)
print("Project checkout:", PROJECT)

## 2. Verify the original inputs and declared change

Attach Notebook 01 and the completed Notebook 05 output. The input check below refuses a different first adapter, bundle, scorer, or incomplete predictions before training starts. Learning rate is the only changed optimization setting; generation monitoring is new instrumentation.


In [ ]:
import hashlib
import shutil

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings before continuing.")
_ = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=True)
INPUT_ROOT = Path(os.environ.get("MATH_BUNDLE_PATH", "/kaggle/input"))
BUNDLE = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_bundle.py"), str(INPUT_ROOT)], text=True).strip())
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/verify_bundle.py"), str(BUNDLE)], check=True)
CONFIG_PATH = PROJECT / "configs/lora_training_lower_lr.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
SOURCE_MANIFEST = json.loads((BUNDLE / "manifest.json").read_text())
BUNDLE_HASH = hashlib.sha256((BUNDLE / "manifest.json").read_bytes()).hexdigest()
CONFIG_HASH = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
RUN_ROOT = Path("/kaggle/working") / f"lora-lower-lr-{BUNDLE_HASH[:12]}-{CONFIG_HASH[:8]}-{CODE_COMMIT[:8]}"
SMOKE_OUTPUT, TRAIN_OUTPUT = RUN_ROOT / "smoke", RUN_ROOT / "full"
print("Bundle:", BUNDLE)
print("Pinned model:", SOURCE_MANIFEST["model"])
print("Original data counts:", SOURCE_MANIFEST["counts"])
print("Training settings:", json.dumps(CONFIG, indent=2))
print("Output:", RUN_ROOT)

PREVIOUS = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_results.py"),
    os.environ.get("MATH_VALIDATION_PATH", "/kaggle/input"), "--kind", "validation"], text=True).strip())
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/compare_training_trial.py"),
    "--bundle", str(BUNDLE), "--previous", str(PREVIOUS), "--verify-only"], check=True, cwd=PROJECT)
print("Verified Notebook 05:", PREVIOUS)


## 3. Prepare the training environment

This adds pinned PEFT and Accelerate packages to the known isolated PyTorch setup. The Kaggle kernel packages are not replaced. The shared setup avoids Kaggle's missing `ensurepip` problem.

The existing `wrapt` startup warning and `torch_dtype` deprecation warning may appear. CUDA execution and the following contract tests must pass before training.


In [ ]:
ENVIRONMENT = Path(tempfile.mkdtemp(prefix="math-training-env-"))
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/setup_environment.py"),
                    "--environment", str(ENVIRONMENT), "--profile", "training"], check=True)
PYTHON = ENVIRONMENT / "bin" / "python"
TRAIN_ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
probe = """
import torch, peft, accelerate
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable: check Kaggle GPU settings.')
x = torch.ones((32, 32), device='cuda', dtype=torch.float16)
assert (x @ x).sum().item() == 32768
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))
print('PEFT:', peft.__version__, 'Accelerate:', accelerate.__version__)
print('Visible GPUs for training:', torch.cuda.device_count())
"""
_ = subprocess.run([str(PYTHON), "-I", "-c", probe], check=True, env=TRAIN_ENV)


## 4. Check training and monitoring on CPU

Tiny local models test masking, loss, checkpoint recovery, and adapter reload. Additional tests check validation-only selection, discovery, restoration of random/model state after errors, and **identical adapter updates with and without epoch generation**. These checks do not require remote model downloads.


In [ ]:
for test_file in ("test_training.py", "test_training_monitor.py", "test_training_trial.py"):
    _ = subprocess.run([str(PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", test_file, "-b", "-v"],
        cwd=PROJECT, check=True, env=TRAIN_ENV)


## 5. Audit training and validation tokenization before loading weights

Each example is encoded as separately tokenized prompt + response + EOS. Prompt labels and padding are `-100`, so the loss trains the solution and final answer. Prompt text remains visible as context. There is no packing or truncation.

Every training token count must match Notebook 01's saved audit. Validation solutions are teacher-forced in the same zero-shot text format used for training. This is different from generating answers: it measures how well the model predicts the supplied correct solution tokens.

All original training/validation rows are used in the full run. If a sequence exceeds 1024 tokens, stop and inspect it rather than silently dropping or shortening it.


In [ ]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/train_lora.py"),
                    "--bundle", str(BUNDLE), "--config", str(CONFIG_PATH), "--stage", "full",
                    "--output", str(RUN_ROOT / "audit"), "--audit-only"],
                    check=True, cwd=PROJECT, env=TRAIN_ENV)
AUDIT = json.loads((RUN_ROOT / "audit/data_audit.json").read_text())
print({key: AUDIT[key] for key in ("train_rows", "validation_rows", "max_train_tokens", "max_validation_tokens", "effective_batch_size", "planned_optimizer_updates")})


## 6. Run the GPU smoke check

Run two updates on 64 training rows, then discard this adapter. The generation check uses two validation questions at the initial, epoch-end, and selected states. This checks the real GPU monitoring path, not maths quality. Full training starts again from the original model.


In [ ]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/train_lora.py"),
                    "--bundle", str(BUNDLE), "--config", str(CONFIG_PATH), "--stage", "smoke",
                    "--output", str(SMOKE_OUTPUT)], check=True, cwd=PROJECT, env=TRAIN_ENV)
SMOKE = json.loads((SMOKE_OUTPUT / "summary.json").read_text())
if not SMOKE["health_passed"]:
    raise RuntimeError("Training smoke check failed; inspect the saved outputs.")
print("Smoke check:", json.dumps(SMOKE, indent=2))


## 7. Train the fresh lower-learning-rate adapter

Use the same three-epoch training schedule and lowest validation response-token loss selection. After each epoch, generate and save answers on the fixed 50 validation questions. Log correctness, parseability, repetition, cap hits, and paired changes versus the original model. Training mode, cache settings, and random state are restored afterward.

The final `selected` observation evaluates the loss-selected checkpoint, which can differ from the final epoch. It is generated again deliberately. Do not choose an earlier epoch based on these observations and call it the same selection experiment.

If interrupted, rerun this cell in the same session. Checkpointed training state resumes; observations for an uncheckpointed epoch may be recomputed. Do not change code, packages, settings, or inputs when resuming.


In [ ]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/train_lora.py"),
                    "--bundle", str(BUNDLE), "--config", str(CONFIG_PATH), "--stage", "full",
                    "--smoke-output", str(SMOKE_OUTPUT), "--output", str(TRAIN_OUTPUT)],
                    check=True, cwd=PROJECT, env=TRAIN_ENV)
SUMMARY = json.loads((TRAIN_OUTPUT / "summary.json").read_text())
print(json.dumps(SUMMARY, indent=2))


## 8. Compare with the first adapter and save the complete output

The CPU report verifies Notebook 05 and compares its original/first-adapter answers against the new loss-selected adapter on the same 50 questions. It reports paired gains and regressions and checks whether the repeated original-model answers match. Differences require investigation before attributing the result to learning rate.

Keep the complete ZIP, including raw epoch predictions, selected adapter, identities, and checkpoints. Share the comparison summary and executed notebook for review. No full benchmark or additional training starts automatically.


In [ ]:
from IPython.display import HTML, display
COMPARISON = RUN_ROOT / "comparison"
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/compare_training_trial.py"),
    "--bundle", str(BUNDLE), "--previous", str(PREVIOUS), "--training", str(TRAIN_OUTPUT),
    "--output", str(COMPARISON)], check=True, cwd=PROJECT, env=TRAIN_ENV)
for output in (SMOKE_OUTPUT, TRAIN_OUTPUT, COMPARISON):
    result = json.loads((output / "result_manifest.json").read_text())
    for name, expected in result["files_sha256"].items():
        if hashlib.sha256((output / name).read_bytes()).hexdigest() != expected:
            raise RuntimeError(f"Result checksum mismatch: {name}")
display(HTML((TRAIN_OUTPUT / "report.html").read_text()))
display(HTML((COMPARISON / "report.html").read_text()))
ARCHIVE = Path(shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT))
print("Selected adapter:", TRAIN_OUTPUT / "adapter")
print("Comparison summary:", COMPARISON / "summary.json")
print("Complete ZIP:", ARCHIVE)
print("Save Version with outputs. Review validation results before another experiment.")
